# Анализ влияния цен золота на котировки золотодобывающих компаний

## Цель исследования
Изучить характер и степень влияния цен на физическое золото на котировки индексов золотодобывающих компаний с использованием современных эконометрических методов.

## 1. Загрузка библиотек

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import os
import warnings
warnings.filterwarnings('ignore')

# Настройка стиля визуализации
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (14, 7)
plt.rcParams['font.size'] = 12

## 2. Загрузка и первичная обработка данных

In [2]:
def clean_european_number(x):
    """Преобразует европейский формат чисел (2.499,97) в float (2499.97)"""
    if pd.isna(x) or x == '' or x == '-':
        return np.nan
    # Удаляем пробелы, заменяем '.' на '' (тысячи), ',' на '.' (десятичный)
    x = str(x).replace(' ', '').replace('%', '').strip()
    if x == '':
        return np.nan
    try:
        # Замена: 2.499,97 → 2499.97
        x = x.replace('.', '').replace(',', '.')
        return float(x)
    except:
        return np.nan

def load_and_clean_data(file_path, asset_name):
    """
    Загрузка и очистка данных из CSV файла с европейским форматом чисел.
    
    Параметры:
    -----------
    file_path : str
        Путь к файлу с данными
    asset_name : str
        Название актива для переименования колонки цены
        
    Возвращает:
    -----------
    pd.DataFrame
        Очищенный датафрейм с колонками: Date, Price, Open, High, Low, Volume, Change_pct
    """
    try:
        # Загрузка с правильной кодировкой
        df = pd.read_csv(file_path, encoding='utf-8-sig', sep=',')
        
        # Очистка названий колонок от лишних пробелов
        df.columns = df.columns.str.strip()
        
        # Проверка наличия необходимых колонок
        required_cols = ['Дата', 'Цена', 'Откр.', 'Макс.', 'Мин.', 'Объём', 'Изм. %']
        if not all(col in df.columns for col in required_cols):
            print(f"⚠️ Внимание: В файле {file_path} отсутствуют необходимые колонки. Используются доступные данные.")
            # Попытка найти альтернативные названия колонок
            col_mapping = {
                'Цена': ['Цена', 'Цена', 'Цена', 'Цена'],
                'Откр.': ['Откр.', 'Открытие', 'Open'],
                'Макс.': ['Макс.', 'Максимум', 'High'],
                'Мин.': ['Мин.', 'Минимум', 'Low'],
                'Объём': ['Объём', 'Объем', 'Volume'],
                'Изм. %': ['Изм. %', 'Изменение %', 'Change %']
            }
        
        # Преобразование даты
        df['Date'] = pd.to_datetime(df['Дата'], format='%d.%m.%Y', errors='coerce')
        
        # Очистка числовых колонок с использованием европейского формата
        df['Price'] = df['Цена'].apply(clean_european_number)
        df['Open'] = df['Откр.'].apply(clean_european_number)
        df['High'] = df['Макс.'].apply(clean_european_number)
        df['Low'] = df['Мин.'].apply(clean_european_number)
        df['Volume'] = df['Объём'].apply(clean_european_number)
        df['Change_pct'] = df['Изм. %'].apply(clean_european_number)
        
        # Выбор необходимых колонок и сортировка по дате
        df_clean = df[['Date', 'Price', 'Open', 'High', 'Low', 'Volume', 'Change_pct']].copy()
        df_clean = df_clean.sort_values('Date').reset_index(drop=True)
        
        # Удаление строк с пропущенными датами или ценами
        df_clean = df_clean.dropna(subset=['Date', 'Price'])
        
        # Переименование колонки цены для единообразия
        df_clean = df_clean.rename(columns={'Price': asset_name})
        
        return df_clean
    
    except Exception as e:
        print(f"❌ Ошибка при загрузке файла {file_path}: {str(e)}")
        return None


In [3]:
# Путь к папке с данными
data_path = 'data2'

# Словарь соответствия файлов и названий активов
files_mapping = {
    'Прошлые данные - XAU_USD.csv': 'Gold',
    'Прошлые данные по GDX.csv': 'GDX',
    'Прошлые данные - Arca Gold BUGS.csv': 'HUI',
    'Прошлые данные по XGD.csv': 'XGD',
    'Прошлые данные - FTSE Gold Mines.csv': 'FTSE_Gold',
    'Прошлые данные - Philadelphia Gold_Silver.csv': 'XAU_Index',
    'Прошлые данные - MVIS Global Junior Gold Miners TR Net.csv': 'MVG',
    'Прошлые данные - Фьючерс на нефть Brent.csv': 'Brent',
    'Прошлые данные - Фьючерсы на нефть WTI.csv': 'WTI',
    'Прошлые данные - Индекс USD.csv': 'USD_Index',
    'Прошлые данные - CBOE Volatility Index.csv': 'VIX',
    'Прошлые данные - S&P 500.csv': 'SP500',
    'Прошлые данные - MSCI ACWI IMI.csv': 'MSCI_ACWI',
    'Прошлые данные - CAD_USD.csv': 'CAD_USD'
}

# Загрузка всех данных в словарь
data_dict = {}

for filename, asset_name in files_mapping.items():
    file_path = os.path.join(data_path, filename)
    if os.path.exists(file_path):
        print(f"Загрузка данных: {asset_name} из файла {filename}...")
        df = load_and_clean_data(file_path, asset_name)
        if df is not None:
            data_dict[asset_name] = df
            print(f"  ✓ Успешно загружено {len(df)} записей, период: {df['Date'].min().date()} - {df['Date'].max().date()}")
        else:
            print(f"  ✗ Не удалось загрузить данные для {asset_name}")
    else:
        print(f"⚠️ Файл не найден: {filename}")

print(f"\n✅ Загружено {len(data_dict)} наборов данных")


Загрузка данных: Gold из файла Прошлые данные - XAU_USD.csv...
  ✓ Успешно загружено 4902 записей, период: 2007-02-19 - 2025-12-16
Загрузка данных: GDX из файла Прошлые данные по GDX.csv...
  ✓ Успешно загружено 4768 записей, период: 2007-02-20 - 2025-12-16
Загрузка данных: HUI из файла Прошлые данные - Arca Gold BUGS.csv...
  ✓ Успешно загружено 3914 записей, период: 2010-05-18 - 2025-12-15
Загрузка данных: XGD из файла Прошлые данные по XGD.csv...
  ✓ Успешно загружено 4727 записей, период: 2007-02-16 - 2025-12-16
Загрузка данных: FTSE_Gold из файла Прошлые данные - FTSE Gold Mines.csv...
  ✓ Успешно загружено 4919 записей, период: 2007-02-19 - 2025-12-15
Загрузка данных: XAU_Index из файла Прошлые данные - Philadelphia Gold_Silver.csv...
  ✓ Успешно загружено 4741 записей, период: 2007-02-20 - 2025-12-16
Загрузка данных: MVG из файла Прошлые данные - MVIS Global Junior Gold Miners TR Net.csv...
  ✓ Успешно загружено 3156 записей, период: 2014-05-13 - 2025-12-12
Загрузка данных: Bren

## 3. Создание отдельных датафреймов для каждой пары "золото-индекс"

In [4]:
# Список индексов золотодобытчиков для анализа
gold_miner_indices = ['GDX', 'HUI', 'XGD', 'FTSE_Gold', 'XAU_Index', 'MVG']

# Словарь для хранения отдельных датафреймов
pair_dfs = {}

# Создаем отдельный датафрейм для каждой пары "золото-индекс"
for index_name in gold_miner_indices:
    if index_name in data_dict and 'Gold' in data_dict:
        # Внутреннее объединение ТОЛЬКО по общим торговым дням
        df_pair = pd.merge(
            data_dict['Gold'][['Date', 'Gold']],
            data_dict[index_name][['Date', index_name]],
            on='Date',
            how='inner'  # КЛЮЧЕВОЙ МОМЕНТ: только дни, когда торговались оба актива
        )
        
        # Добавляем макропеременные через ffill (только для дней, когда есть золото+индекс)
        macro_vars = ['USD_Index', 'VIX', 'SP500', 'Brent']
        for var in macro_vars:
            if var in data_dict:
                # Объединяем с макропеременной
                df_pair = pd.merge(
                    df_pair,
                    data_dict[var][['Date', var]],
                    on='Date',
                    how='left'
                )
                # Заполняем пропуски методом forward fill
                df_pair[var] = df_pair[var].fillna(method='ffill')
        
        # Удаляем оставшиеся пропуски (если макропеременная отсутствует в начале периода)
        df_pair = df_pair.dropna()
        
        pair_dfs[index_name] = df_pair
        
        print(f"\n{index_name}:")
        print(f"  Период: {df_pair['Date'].min().date()} - {df_pair['Date'].max().date()}")
        print(f"  Количество наблюдений: {len(df_pair)}")
        print(f"  Пропусков: {df_pair.isnull().sum().sum()} (0% после очистки)")

print("\n✅ Созданы отдельные датафреймы для каждой пары 'золото-индекс'")


GDX:
  Период: 2007-02-20 - 2025-12-16
  Количество наблюдений: 4740
  Пропусков: 0 (0% после очистки)

HUI:
  Период: 2010-05-18 - 2025-12-15
  Количество наблюдений: 3914
  Пропусков: 0 (0% после очистки)

XGD:
  Период: 2007-02-20 - 2025-12-16
  Количество наблюдений: 4725
  Пропусков: 0 (0% после очистки)

FTSE_Gold:
  Период: 2007-02-20 - 2025-12-15
  Количество наблюдений: 4869
  Пропусков: 0 (0% после очистки)

XAU_Index:
  Период: 2007-02-20 - 2025-12-16
  Количество наблюдений: 4738
  Пропусков: 0 (0% после очистки)

MVG:
  Период: 2014-05-13 - 2025-12-12
  Количество наблюдений: 3003
  Пропусков: 0 (0% после очистки)

✅ Созданы отдельные датафреймы для каждой пары 'золото-индекс'


## 4. Расчет доходностей и технических индикаторов

In [5]:
# Словарь для хранения датафреймов с доходностями
returns_dfs = {}

for index_name, df in pair_dfs.items():
    df_ret = df.copy()
    
    # Расчет логарифмических доходностей ТОЛЬКО для дней без пропусков
    df_ret['Gold_Return'] = np.log(df_ret['Gold'] / df_ret['Gold'].shift(1))
    df_ret['Index_Return'] = np.log(df_ret[index_name] / df_ret[index_name].shift(1))
    
    # Для макропеременных тоже рассчитываем доходности
    if 'USD_Index' in df_ret.columns:
        df_ret['USD_Return'] = np.log(df_ret['USD_Index'] / df_ret['USD_Index'].shift(1))
    if 'Brent' in df_ret.columns:
        df_ret['Brent_Return'] = np.log(df_ret['Brent'] / df_ret['Brent'].shift(1))
    if 'SP500' in df_ret.columns:
        df_ret['SP500_Return'] = np.log(df_ret['SP500'] / df_ret['SP500'].shift(1))
    
    # Удаляем первую строку (где доходность = NaN) и оставшиеся пропуски
    df_ret = df_ret.dropna().reset_index(drop=True)
    
    returns_dfs[index_name] = df_ret
    
    print(f"\n{index_name}:")
    print(f"  Период доходностей: {df_ret['Date'].min().date()} - {df_ret['Date'].max().date()}")
    print(f"  Количество наблюдений: {len(df_ret)}")
    print(f"  Средняя доходность золота: {df_ret['Gold_Return'].mean()*100:.4f}%")
    print(f"  Средняя доходность индекса: {df_ret['Index_Return'].mean()*100:.4f}%")
    print(f"  Корреляция доходностей: {df_ret['Gold_Return'].corr(df_ret['Index_Return']):.4f}")

print("\n✅ Расчет доходностей завершен для всех пар")


GDX:
  Период доходностей: 2007-02-21 - 2025-12-16
  Количество наблюдений: 4739
  Средняя доходность золота: 0.0397%
  Средняя доходность индекса: 0.0158%
  Корреляция доходностей: 0.7557

HUI:
  Период доходностей: 2010-05-19 - 2025-12-15
  Количество наблюдений: 3913
  Средняя доходность золота: 0.0322%
  Средняя доходность индекса: 0.0099%
  Корреляция доходностей: 0.7465

XGD:
  Период доходностей: 2007-02-21 - 2025-12-16
  Количество наблюдений: 4724
  Средняя доходность золота: 0.0398%
  Средняя доходность индекса: 0.0195%
  Корреляция доходностей: 0.7426

FTSE_Gold:
  Период доходностей: 2007-02-21 - 2025-12-15
  Количество наблюдений: 4868
  Средняя доходность золота: 0.0386%
  Средняя доходность индекса: 0.0166%
  Корреляция доходностей: 0.7462

XAU_Index:
  Период доходностей: 2007-02-21 - 2025-12-16
  Количество наблюдений: 4737
  Средняя доходность золота: 0.0397%
  Средняя доходность индекса: 0.0184%
  Корреляция доходностей: 0.7287

MVG:
  Период доходностей: 2014-05-14

In [6]:
returns_dfs['GDX']

,Date,Gold,GDX,USD_Index,VIX,SP500,Brent,Gold_Return,Index_Return,USD_Return,Brent_Return,SP500_Return
0,2007-02-21,679.25,41.57,84.21,10.20,1457.60,59.35,0.030645,0.036501,0.000238,0.023354,-0.001440
1,2007-02-22,677.75,41.80,84.32,10.18,1456.40,60.62,-0.002211,0.005518,0.001305,0.021173,-0.000824
2,2007-02-23,682.85,41.83,84.07,10.58,1451.20,60.88,0.007497,0.000717,-0.002969,0.004280,-0.003577
3,2007-02-26,687.35,42.35,83.93,11.15,1449.40,61.33,0.006568,0.012355,-0.001667,0.007364,-0.001241
4,2007-02-27,680.00,39.35,83.48,18.31,1399.00,61.36,-0.010751,-0.073472,-0.005376,0.000489,-0.035392
...,...,...,...,...,...,...,...,...,...,...,...,...
4734,2025-12-10,4228.55,83.32,98.79,15.77,6886.68,62.21,0.004506,0.016701,-0.004343,0.004350,0.006727
4735,2025-12-11,4283.28,86.26,98.35,14.85,6901.00,61.28,0.012860,0.034677,-0.004464,-0.015062,0.002077
4736,2025-12-12,4302.43,85.66,98.40,15.74,6827.41,61.12,0.004461,-0.006980,0.000508,-0.002614,-0.010721
4737,2025-12-15,4302.70,84.85,98.31,16.50,6816.51,60.56,0.000063,-0.009501,-0.000915,-0.009205,-0.001598


## Базовая регрессия для каждой пары

In [7]:
import statsmodels.api as sm

print("РЕГРЕССИОННЫЙ АНАЛИЗ ЗАВИСИМОСТИ ИНДЕКСОВ ОТ ЦЕНЫ ЗОЛОТА")
print("="*70)

results = []

for index_name, df in returns_dfs.items():
    print(f"\n{index_name}:")
    print("-" * 70)
    
    # Базовая модель: доходность индекса ~ доходность золота + доходность доллара + доходность нефти
    X = df[['Gold_Return', 'USD_Return', 'Brent_Return']].copy()
    X = sm.add_constant(X)  # добавляем константу
    y = df['Index_Return']
    
    # Оценка модели
    model = sm.OLS(y, X).fit()
    
    # Сохраняем результаты
    results.append({
        'index': index_name,
        'gold_beta': model.params['Gold_Return'],
        'usd_beta': model.params['USD_Return'],
        'brent_beta': model.params['Brent_Return'],
        'r_squared': model.rsquared,
        'f_stat': model.fvalue,
        'p_value': model.f_pvalue,
        'n_obs': len(df)
    })
    
    # Вывод результатов
    print(model.summary())
    print(f"\nЗолотая бета (чувствительность к золоту): {model.params['Gold_Return']:.4f}")
    print(f"R²: {model.rsquared:.4f}")
    print(f"F-statistic: {model.fvalue:.2f} (p-value: {model.f_pvalue:.4f})")

# Создаем сводную таблицу результатов
results_df = pd.DataFrame(results)
print("\n\nСВОДНАЯ ТАБЛИЦА РЕЗУЛЬТАТОВ")
print("="*70)
print(results_df[['index', 'gold_beta', 'usd_beta', 'brent_beta', 'r_squared', 'n_obs']].to_string(index=False))

РЕГРЕССИОННЫЙ АНАЛИЗ ЗАВИСИМОСТИ ИНДЕКСОВ ОТ ЦЕНЫ ЗОЛОТА

GDX:
----------------------------------------------------------------------
                            OLS Regression Results                            
Dep. Variable:           Index_Return   R-squared:                       0.593
Model:                            OLS   Adj. R-squared:                  0.592
Method:                 Least Squares   F-statistic:                     2297.
Date:                Wed, 15 Apr 2026   Prob (F-statistic):               0.00
Time:                        17:12:06   Log-Likelihood:                 12725.
No. Observations:                4739   AIC:                        -2.544e+04
Df Residuals:                    4735   BIC:                        -2.542e+04
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.